# 03. Uplift Modeling

In [ ]:

import os, sys
from pathlib import Path
import pandas as pd

sys.path.append(os.path.abspath(".."))
from src.utils import load_config
from src.uplift_models import train_two_models, predict_uplift
from src.evaluation import uplift_at_k, auuc

config = load_config("../configs/config.yaml")
cols = config["columns"]
processed_dir = Path("..") / config["paths"]["processed_dir"]


In [ ]:

train_feat = pd.read_parquet(processed_dir / "train_features.parquet")
test_feat = pd.read_parquet(processed_dir / "test_features.parquet")
print(train_feat.shape, test_feat.shape)


In [ ]:

model_t, model_c, features = train_two_models(train_feat, config)
uplift_pred_train = predict_uplift(model_t, model_c, train_feat, features)
print("uplift@30%:", uplift_at_k(train_feat[cols["target"]].values, train_feat[cols["treatment"]].values, uplift_pred_train, 0.3))
print("AUUC:", auuc(train_feat[cols["target"]].values, train_feat[cols["treatment"]].values, uplift_pred_train))


In [ ]:

submission = test_feat[[cols["client_id"]]].copy()
submission["uplift_score"] = predict_uplift(model_t, model_c, test_feat, features)
submission.head()
